<img src="https://www.th-koeln.de/img/logo.svg" style="float:right;" width="200">

# 13th exercise: <font color="#C70039">Q-learning in Gymnasium — FrozenLake</font>

* Course: AML  
* Lecturer: <a href="https://www.gernotheisenberg.de/">Gernot Heisenberg</a>
* Author of notebook: <a href="https://www.gernotheisenberg.de/">Gernot Heisenberg</a>
* Date: 03.09.2026

---

**GENERAL NOTE 1**: 
Please make sure you are reading the entire notebook, since it contains a lot of information on your tasks (e.g. regarding the set of certain parameters or a specific computational trick), and the written mark downs as well as comments contain a lot of information on how things work together as a whole. 

**GENERAL NOTE 2**: 
* Please, when commenting source code, just use English language only. 
* When describing an observation please use English language, too.
* This applies to all exercises throughout this course.

---------------------------------

### <font color="FFC300">LEARNING OBJECTIVES</font>:

After this exercise, you can transfer tabular Q-learning to a Gymnasium environment, use its `reset` and `step` API correctly, distinguish terminal and truncated episodes, evaluate a learned policy, and analyse the effect of stochastic transitions and sparse rewards.

### <font color="green">SAMPLE SOLUTION</font>:

This notebook is a completed copy of the corresponding exercise Ex13. It shows one valid implementation and concise explanations of the design choices. The explanations focus on why each step is needed.

## From Exercise 12 to FrozenLake

Exercise 12 used an explicit and deterministic transition graph. Here Gymnasium supplies the transitions. The agent crosses a 4×4 frozen lake: `S` marks the start, `F` safe ice, `H` a hole, and `G` the goal. Reaching `G` yields reward 1; all other transitions yield reward 0.

The lake is slippery. An intended action may therefore lead to a different neighbouring state. This makes the environment **stochastic** and adds uncertainty to the learning problem.

<img src="./images/FrozenLake.States.Rewards.png" width="800">

## Setup

Run the next cell in Google Colab if needed. On a local installation, install `gymnasium[toy-text]` once before running the notebook.

In [ ]:
import os
import subprocess
import sys

if "google.colab" in sys.modules:
    repository = "/content/AML"
    if not os.path.isdir(repository):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/gheisenberg/AML.git", repository], check=True)
    os.chdir(repository)
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "gymnasium[toy-text]"], check=True)

import gymnasium as gym
import numpy as np

print(f"Gymnasium version: {gym.__version__}")

## Create and inspect the environment

Gymnasium returns `(observation, info)` from `reset()` and `(observation, reward, terminated, truncated, info)` from `step(action)`. `terminated` means that a terminal state was reached. `truncated` means that an external limit ended the episode. Both end an episode, but neither should bootstrap a future Q-value.

The action indices are `0 = left`, `1 = down`, `2 = right`, and `3 = up`.

In [ ]:
env = gym.make("FrozenLake-v1", is_slippery=True)
env.action_space.seed(1)
rng = np.random.default_rng(1)

print("Number of states:", env.observation_space.n)
print("Number of actions:", env.action_space.n)
print("Action meanings:", env.unwrapped.get_action_meanings())

## Hyperparameters and Q-table

Keep the seed and all but one hyperparameter constant when conducting an experiment. This makes differences in the results interpretable.

In [ ]:
num_episodes = 20_000
max_steps_per_episode = 100
learning_rate = 0.1
discount_rate = 0.99
exploration_rate = 1.0
max_exploration_rate = 1.0
min_exploration_rate = 0.01
exploration_decay_rate = 0.001

q_table = np.zeros((env.observation_space.n, env.action_space.n), dtype=float)

## <font color="FFC300">Task 1</font> — transfer the training loop

Implement `train_q_learning`. Reuse the epsilon-greedy principle and Q-learning update from Exercise 12, but adapt them to the Gymnasium API.

Your implementation must:

1. reset the environment at the start of every episode;
2. choose between a random action and the greedy action;
3. combine `terminated` and `truncated` into one episode-finished condition;
4. use future value zero for a finished episode;
5. store the total reward per episode; and
6. decay epsilon after every episode.

Use `rng` for the epsilon decision so the experiment remains reproducible.

### Sample answer — Task 1

The loop retains the Q-learning update from Exercise 12, but Gymnasium provides the transition rather than an adjacency matrix. `terminated or truncated` is used as the episode boundary. In both cases the bootstrap value is zero, because the environment will not provide a following decision within that episode.

In [ ]:
def train_q_learning(environment):
    global exploration_rate
    rewards_all_episodes = []

    for episode in range(num_episodes):
        state, info = environment.reset(seed=episode)
        episode_reward = 0.0

        for _ in range(max_steps_per_episode):
            if rng.random() < exploration_rate:
                action = environment.action_space.sample()
            else:
                action = int(np.argmax(q_table[state]))

            new_state, reward, terminated, truncated, info = environment.step(action)
            episode_finished = terminated or truncated
            future_value = 0.0 if episode_finished else np.max(q_table[new_state])
            temporal_difference = reward + discount_rate * future_value - q_table[state, action]
            q_table[state, action] += learning_rate * temporal_difference

            state = new_state
            episode_reward += reward
            if episode_finished:
                break

        exploration_rate = min_exploration_rate + (
            max_exploration_rate - min_exploration_rate
        ) * np.exp(-exploration_decay_rate * episode)
        rewards_all_episodes.append(episode_reward)

    return np.asarray(rewards_all_episodes)

rewards_all_episodes = train_q_learning(env)

## <font color="FFC300">Task 2</font> — evaluate learning progress

FrozenLake returns reward 1 only when the agent reaches the goal. The mean episode reward is therefore the empirical success rate. Complete the reporting cell and discuss whether the rate improves during training.

Do not interpret a single episode as evidence of learning; use blocks of many episodes.

### Sample answer — Task 2

The reporting function computes the mean binary reward in blocks of 1,000 episodes, so each value is a success rate. This aggregation is necessary because individual episodes are dominated by the sparse reward and by random slipping. A rising block average is evidence that the policy is improving.

In [ ]:
def report_success_rates(rewards, block_size=1_000):
    for start in range(0, len(rewards), block_size):
        block = rewards[start:start + block_size]
        print(f"Episodes {start + 1:5d}-{start + len(block):5d}: success rate = {block.mean():.3f}")

# report_success_rates(rewards_all_episodes)

## <font color="FFC300">Task 3</font> — test the greedy policy

Complete the evaluator. It must act greedily from the learned Q-table, perform no exploration, and report the mean reward across many fresh episodes. Use the same slippery environment for evaluation.

Compare this result with the final training blocks. Explain why the two figures can differ.

### Sample answer — Task 3

Evaluation uses only `argmax(table[state])`, so it measures the learned greedy policy rather than the exploratory training behaviour. The final training block can differ because training still uses a non-zero epsilon, and because the slippery environment remains stochastic during both training and evaluation.

In [ ]:
def evaluate_greedy_policy(environment, table, episodes=1_000):
    successful_episodes = 0

    for episode in range(episodes):
        state, info = environment.reset(seed=100_000 + episode)
        for _ in range(max_steps_per_episode):
            action = int(np.argmax(table[state]))
            state, reward, terminated, truncated, info = environment.step(action)
            if terminated or truncated:
                successful_episodes += int(reward == 1)
                break

    return successful_episodes / episodes

evaluation_environment = gym.make("FrozenLake-v1", is_slippery=True)
print(f"Greedy-policy success rate: {evaluate_greedy_policy(evaluation_environment, q_table):.3f}")
evaluation_environment.close()

## <font color="FFC300">Task 4</font> — experimental report

Create a compact test plan with at least four runs. Change one variable at a time, record the final evaluation success rate, and explain the result. Include the learning rate, discount rate, exploration decay rate, and the number of episodes.

Then create `FrozenLake-v1` with `is_slippery=False`, repeat one configuration, and compare it with the slippery case. Explain the difference using the terms *stochastic transitions* and *sparse rewards*.

Finally, print the Q-table and identify one state for which a seemingly best action still cannot guarantee success.

### Sample answer — Task 4

A suitable test plan varies one parameter at a time, for example: baseline; lower learning rate; lower discount rate; slower epsilon decay; and more episodes. Use the same seeds and report the greedy-policy success rate for each run. With `is_slippery=False`, the intended action always succeeds, so the environment is deterministic and a successful path is learned more reliably than in the slippery sparse-reward setting. Exact rates depend on the selected parameters and Gymnasium version.

In [ ]:
report_success_rates(rewards_all_episodes)
print("Learned Q-table:")
print(np.round(q_table, 3))
env.close()

## <font color="FFC300">Task 5</font> — Reflection

1. Which part of the Q-learning algorithm is unchanged from Exercise 12?
2. Which details are specific to Gymnasium?
3. Why must terminal transitions not use a future Q-value?
4. Why does a deterministic version of FrozenLake not make the learned policy automatically transferable to the slippery version?

### Sample answer — Task 5

1. Action selection, the Bellman update, and epsilon decay are unchanged from Exercise 12.
2. `reset`, `step`, `terminated`, and `truncated` are Gymnasium-specific interface details.
3. A terminal transition has no successor decision, so a future Q-value would be invalid.
4. Slipping changes transition probabilities; a policy learned for deterministic movement is therefore not automatically optimal under stochastic movement.